<a href="https://colab.research.google.com/github/leenAbdulaziz-glitch/Recommendation-Scoring-Model/blob/main/Amazon_Review_torch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Summary: Amazon Fine Food Reviews Sentiment Analysis
## Dataset Overview
The dataset consists of reviews of fine foods from Amazon. It includes user information, product ratings, and plaintext reviews. In this project, we focused on:

- **Target**: Predicting user satisfaction (Positive vs. Negative) based on the Score column.

- **Size**: A sampled subset of 5,000 records to ensure efficient processing and high-quality feature extraction.

In [ ]:
# Since I dealing with text which may vary in length or require special processing,
# PyTorch allows you to easily modify the model's behavior during training.

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import kagglehub
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler

In [2]:
# # data
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv")).dropna()
# Selecting  features
df = df[["UserId", "ProductId", "Score", "Summary", "Text"]].dropna()
df = df.sample(1000, random_state=42).reset_index(drop=True)

100%|██████████| 242M/242M [00:02<00:00, 87.9MB/s]

Extracting files...


In [3]:
df.head()

,UserId,ProductId,Score,Summary,Text
0,A2R2GG6WW0J56R,B001PO7FIU,5,Avocado Oil,"Avocado oil beats olive oil for dressings, any..."
1,A3SJSHRFGEV49,B001M08YZA,4,"Healthy, tastes-oky","New years resolution season and all, bought pr..."
2,AIDPKNFYTTJ9Y,B002O19HWU,1,Peaches from China :-(,I was pretty sad when I got these peaches (whi...
3,A3LWC21A3VEVS3,B000CQG8KS,5,Perhaps Brewing Wrong?,I find that most (not all) people in North Ame...
4,A2A8ANYRNWSKP1,B001IZGC58,5,Works Great!,"Love that this is recycled, works just as well..."


In [4]:
# Label Encoding
user_enc = LabelEncoder()
item_enc = LabelEncoder()
df['user_idx'] = user_enc.fit_transform(df['UserId'])
df['item_idx'] = item_enc.fit_transform(df['ProductId'])

In [5]:
# target and text
# Preparing target labels and combined text feature
# Label 1 for positive (Score >= 4), Label 0 for negative
df["label"] = (df["Score"] >= 4).astype(int)
df["review_text"] = df["Summary"] + " " + df["Text"]

In [6]:
# # # Text Vectorization using TF-IDF with Unigrams and Bigrams
# Term Frequency-Inverse Document Frequency
tfidf = TfidfVectorizer(max_features=500, stop_words="english")
X_text = tfidf.fit_transform(df["review_text"]).toarray()

In [7]:
# combined x
user_feat = df["user_idx"].values.reshape(-1, 1)
item_feat = df["item_idx"].values.reshape(-1, 1)
X_combined = np.hstack((user_feat, item_feat, X_text))
y = df["label"].values

In [8]:
# Scaling
# Standardizing features for model stability
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

In [9]:
# Split into Training
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [10]:
# # Baseline Model (Logistic Regression)
baseline_model = LogisticRegression()
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_preds)

In [11]:

print(classification_report(y_test, baseline_preds))

              precision    recall  f1-score   support

           0       0.46      0.48      0.47        46
           1       0.84      0.83      0.84       154

    accuracy                           0.75       200
   macro avg       0.65      0.65      0.65       200
weighted avg       0.75      0.75      0.75       200



In [12]:
# Given that PyTorch cannot perform calculations or update weights using NumPy,
# the Tensor structure is needed so that PyTorch can understand it.
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).view(-1, 1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).view(-1, 1)

In [13]:
# # Deep Learning Model
class RecModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = RecModel(X_train.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [14]:
# training
epochs = 50
train_losses = []

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


Epoch 1/50 - Loss: 0.6801
Epoch 2/50 - Loss: 0.6542
Epoch 3/50 - Loss: 0.6311
Epoch 4/50 - Loss: 0.6060
Epoch 5/50 - Loss: 0.5833
Epoch 6/50 - Loss: 0.5627
Epoch 7/50 - Loss: 0.5416
Epoch 8/50 - Loss: 0.5189
Epoch 9/50 - Loss: 0.5002
Epoch 10/50 - Loss: 0.4808
Epoch 11/50 - Loss: 0.4594
Epoch 12/50 - Loss: 0.4405
Epoch 13/50 - Loss: 0.4203
Epoch 14/50 - Loss: 0.4007
Epoch 15/50 - Loss: 0.3834
Epoch 16/50 - Loss: 0.3617
Epoch 17/50 - Loss: 0.3448
Epoch 18/50 - Loss: 0.3268
Epoch 19/50 - Loss: 0.3080
Epoch 20/50 - Loss: 0.2914
Epoch 21/50 - Loss: 0.2746
Epoch 22/50 - Loss: 0.2586
Epoch 23/50 - Loss: 0.2453
Epoch 24/50 - Loss: 0.2308
Epoch 25/50 - Loss: 0.2167
Epoch 26/50 - Loss: 0.2054
Epoch 27/50 - Loss: 0.1941
Epoch 28/50 - Loss: 0.1824
Epoch 29/50 - Loss: 0.1701
Epoch 30/50 - Loss: 0.1620
Epoch 31/50 - Loss: 0.1520
Epoch 32/50 - Loss: 0.1415
Epoch 33/50 - Loss: 0.1331
Epoch 34/50 - Loss: 0.1260
Epoch 35/50 - Loss: 0.1165
Epoch 36/50 - Loss: 0.1081
Epoch 37/50 - Loss: 0.1024
Epoch 38/5

In [15]:
model.eval()
with torch.no_grad():
    raw_preds = model(X_test_t)
    deep_preds = (raw_preds > 0.5).int().numpy()
    deep_acc = accuracy_score(y_test, deep_preds)

In [16]:
# COMPARISON
print("\n" + "="*30)
print("RESULTS COMPARISON")
print("="*30)
print(f"Baseline (Logistic Regression) Accuracy: {baseline_acc:.4f}")
print(f"Deep Learning Accuracy: {deep_acc:.4f}")
print("\nDetailed Report for Deep Learning:")
print(classification_report(y_test, deep_preds))


RESULTS COMPARISON
Baseline (Logistic Regression) Accuracy: 0.7500
Deep Learning Accuracy: 0.8000

Detailed Report for Deep Learning:
              precision    recall  f1-score   support

           0       0.58      0.48      0.52        46
           1       0.85      0.90      0.87       154

    accuracy                           0.80       200
   macro avg       0.72      0.69      0.70       200
weighted avg       0.79      0.80      0.79       200



In [18]:
def predict_new_review(review_text, user_id_idx=0, prod_id_idx=0):
    # Process text, stack with metadata, scale, and predict
    text_feat = tfidf.transform([review_text]).toarray()
    meta_feat = np.array([[user_id_idx, prod_id_idx]])
    final_feat = np.hstack((meta_feat, text_feat))
    final_scaled = scaler.transform(final_feat)

    model.eval()
    with torch.no_grad():
        prob = model(torch.FloatTensor(final_scaled)).item()

    sentiment = 'Positive' if prob > 0.5 else 'Negative'
    print(f"Review: {review_text}\nSentiment: {sentiment} (Prob: {prob:.4f})\n")

# Test Inference
predict_new_review("This food is delicious and healthy!")

Review: This food is delicious and healthy!
Sentiment: Positive (Prob: 1.0000)



In [19]:
old_english_review = "Thou art a merchant of most excellent goods, thy food hath no peer betwixt the seas!"

print("Testing with Old English / Rare Words Context:")
predict_new_review(old_english_review)

# Let's try a very negative one with old words
negative_old_review = "Alas! The quality is naught but woe, a scurvy product indeed."
predict_new_review(negative_old_review)

Testing with Old English / Rare Words Context:
Review: Thou art a merchant of most excellent goods, thy food hath no peer betwixt the seas!
Sentiment: Positive (Prob: 0.9992)

Review: Alas! The quality is naught but woe, a scurvy product indeed.
Sentiment: Positive (Prob: 0.9942)



In [ ]:
# # Plotting Loss and Accuracy curves for analysis
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, color='red')
plt.title("Deep Learning Loss")
plt.xlabel("Epochs")

In [ ]:
#
plt.subplot(1, 2, 2)
plt.bar(["Baseline", "Deep Learning"], [baseline_acc, deep_acc], color=['gray', 'blue'])
plt.title("Model Accuracy Comparison")
plt.ylim(0, 1.0)
plt.show()